# Ministral-3-14B -- Climate generalization pilot

Content-domain + chart-type generalization test (Chapter 7 Limitation 2, Chapter 8 item 4,
supervisor item 7). Solar-vs-wind line chart, fabricated data, climate-framed claim -- see
`climate_pilot/generate_climate_stimuli.py` for the stimulus design rationale. Same protocols
as the main study (baseline single-image like/scroll + logprobs, single-image across the 6
`metrics/realistic` engagement scales + logprobs, full 7x7 paired A/B `metrics` grid), pointed
at `climate_pilot/posts/` instead of `benchmarking/`. 25 posts (not 50/100) -- this is a scoped
pilot, not a full replication.

In [1]:
import sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "tokenizers>=0.22.0,<=0.23.0",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.16.0.dev0 requires tokenizers<=0.23.0,>=0.22.0, but you have tokenizers 0.23.1 which is incompatible.


✅ Done — restart the kernel now


Restart kernel after running the setup cell above.

In [2]:
!nvidia-smi

Tue Aug 18 15:01:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:CF:00.0 Off |                   On |
| N/A   29C    P0            127W /  700W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

In [3]:
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

from pathlib import Path
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "mistralai/Ministral-3-14B-Instruct-2512-BF16"
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto"
).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, fix_mistral_regex=True)
device = next(model.parameters()).device

# see experiments/e1/ministral-3-14b/e1-ministral-3-14b.ipynb for why this is needed
model.generation_config.max_length = None

ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [4]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Configuration: climate pilot, NOT the main benchmarking/ pool ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_climate/  -- shared across all 4 climate-pilot models, so they see the identical 25-image sample
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 25

correct_dir = ROOT_DIR / "climate_pilot/posts/correct/PNGs"
incorrect_dir = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic condition only -- the condition that showed the strongest conformity effect
# in the main study (Section 6.2)
correct_base = ROOT_DIR / "climate_pilot/posts/correct/PNGs/metrics/realistic"
incorrect_base = ROOT_DIR / "climate_pilot/posts/incorrect/PNGs/metrics/realistic"


📋 Loading existing selection from /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/selected_images.json
✅ All selected numbers verified in both correct and incorrect folders.
Selected 25 pairs → 50 images total


In [5]:
from e1_utils.inference_mistral import run_inference_mistral

In [6]:
from e1_utils.e1_optimized import (
    run_e1_baseline_logprobs, run_e1_metrics_logprobs,
    LIKE_CANDIDATES_SINGLE, LIKE_CANDIDATES_YESNO
)

from e1_utils.inference_mistral import run_inference_with_scores_mistral

## Approach 1 -- single image, like/scroll, baseline (0 engagement)

In [7]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json",
              inference_fn=run_inference_mistral)

📋 Resuming — 1 images already processed.
⏭ Skipping: 001_correct
✅ 001_incorrect → like
✅ 002_correct → like
✅ 002_incorrect → like
✅ 003_correct → like
✅ 003_incorrect → like
✅ 004_correct → like
✅ 004_incorrect → like
✅ 005_correct → like
✅ 005_incorrect → like
✅ 006_correct → like
✅ 006_incorrect → like
✅ 007_correct → like
✅ 007_incorrect → like
✅ 008_correct → like
✅ 008_incorrect → like
✅ 009_correct → like
✅ 009_incorrect → like
✅ 010_correct → like
✅ 010_incorrect → like
✅ 011_correct → like
✅ 011_incorrect → like
✅ 012_correct → like
✅ 012_incorrect → like
✅ 013_correct → like
✅ 013_incorrect → like
✅ 014_correct → like
✅ 014_incorrect → like
✅ 015_correct → like
✅ 015_incorrect → like
✅ 016_correct → like
✅ 016_incorrect → like
✅ 017_correct → like
✅ 017_incorrect → like
✅ 018_correct → like
✅ 018_incorrect → like
✅ 019_correct → like
✅ 019_incorrect → like
✅ 020_correct → like
✅ 020_incorrect → like
✅ 021_correct → like
✅ 021_incorrect → like
✅ 022_correct → like
✅ 022_incor

In [8]:
run_e1_baseline_logprobs(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_baseline_logprobs.json", score_fn=run_inference_with_scores_mistral)

✅ 001_correct → like {'like': {'logprob': -11.38238525390625, 'prob_forced_choice': 0.9883127420043056}, 'scroll': {'logprob': -15.81988525390625, 'prob_forced_choice': 0.011687257995694434}}
✅ 001_incorrect → like {'like': {'logprob': -10.584456443786621, 'prob_forced_choice': 0.6076631698328917}, 'scroll': {'logprob': -11.021956443786621, 'prob_forced_choice': 0.3923368301671084}}
✅ 002_correct → like {'like': {'logprob': -11.195083618164062, 'prob_forced_choice': 0.98504291340685}, 'scroll': {'logprob': -15.382583618164062, 'prob_forced_choice': 0.01495708659314999}}
✅ 002_incorrect → like {'like': {'logprob': -10.947345733642578, 'prob_forced_choice': 0.9149009549929797}, 'scroll': {'logprob': -13.322345733642578, 'prob_forced_choice': 0.08509904500702024}}
✅ 003_correct → like {'like': {'logprob': -11.319951057434082, 'prob_forced_choice': 0.9875683491468141}, 'scroll': {'logprob': -15.694951057434082, 'prob_forced_choice': 0.012431650853185818}}
✅ 003_incorrect → like {'like': {'

In [9]:
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")


Single image analysis: e1_results_baseline
=== Summary ===


,metric,value
0,overall_like_rate_%,100.0
1,like_rate_correct_%,100.0
2,like_rate_incorrect_%,100.0


=== Per Image Results ===


,image,variant,prompt,answer
0,001_correct,correct,You are shown a social media post.\nYou can ei...,like
1,001_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
2,002_correct,correct,You are shown a social media post.\nYou can ei...,like
3,002_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
4,003_correct,correct,You are shown a social media post.\nYou can ei...,like
5,003_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
6,004_correct,correct,You are shown a social media post.\nYou can ei...,like
7,004_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like
8,005_correct,correct,You are shown a social media post.\nYou can ei...,like
9,005_incorrect,incorrect,You are shown a social media post.\nYou can ei...,like


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/ministral-3-14b/outputs/e1_analysis_baseline.csv


## Approach 1 variant -- single image, like/scroll, across the 6 `metrics/realistic` engagement scales

In [10]:
run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",
              inference_fn=run_inference_mistral)

✅ 001_correct_10 → like
✅ 001_incorrect_10 → like
✅ 002_correct_10 → like
✅ 002_incorrect_10 → like
✅ 003_correct_10 → like
✅ 003_incorrect_10 → like
✅ 004_correct_10 → like
✅ 004_incorrect_10 → like
✅ 005_correct_10 → like
✅ 005_incorrect_10 → like
✅ 006_correct_10 → like
✅ 006_incorrect_10 → like
✅ 007_correct_10 → like
✅ 007_incorrect_10 → like
✅ 008_correct_10 → like
✅ 008_incorrect_10 → like
✅ 009_correct_10 → like
✅ 009_incorrect_10 → like
✅ 010_correct_10 → like
✅ 010_incorrect_10 → like
✅ 011_correct_10 → like
✅ 011_incorrect_10 → like
✅ 012_correct_10 → like
✅ 012_incorrect_10 → like
✅ 013_correct_10 → like
✅ 013_incorrect_10 → like
✅ 014_correct_10 → like
✅ 014_incorrect_10 → like
✅ 015_correct_10 → like
✅ 015_incorrect_10 → like
✅ 016_correct_10 → like
✅ 016_incorrect_10 → like
✅ 017_correct_10 → like
✅ 017_incorrect_10 → like
✅ 018_correct_10 → like
✅ 018_incorrect_10 → like
✅ 019_correct_10 → like
✅ 019_incorrect_10 → like
✅ 020_correct_10 → like
✅ 020_incorrect_10 → like


In [11]:
run_e1_metrics_logprobs(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_metrics_logprobs.json", score_fn=run_inference_with_scores_mistral)

✅ 001_correct_10 → like {'like': {'logprob': -11.257641792297363, 'prob_forced_choice': 0.9740426428022031}, 'scroll': {'logprob': -14.882641792297363, 'prob_forced_choice': 0.02595735719779685}}
✅ 001_incorrect_10 → like {'like': {'logprob': -10.88640308380127, 'prob_forced_choice': 0.8080672135527632}, 'scroll': {'logprob': -12.32390308380127, 'prob_forced_choice': 0.19193278644723683}}
✅ 002_correct_10 → like {'like': {'logprob': -11.256830215454102, 'prob_forced_choice': 0.9770226300899744}, 'scroll': {'logprob': -15.006830215454102, 'prob_forced_choice': 0.02297736991002561}}
✅ 002_incorrect_10 → like {'like': {'logprob': -11.071856498718262, 'prob_forced_choice': 0.9324533088603709}, 'scroll': {'logprob': -13.696856498718262, 'prob_forced_choice': 0.0675466911396291}}
✅ 003_correct_10 → like {'like': {'logprob': -11.31857681274414, 'prob_forced_choice': 0.9688561694652216}, 'scroll': {'logprob': -14.75607681274414, 'prob_forced_choice': 0.031143830534778462}}
✅ 003_incorrect_10 →

In [12]:
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")


Metrics single image analysis: e1_results_metrics
=== Overall Summary ===


,metric,value
0,overall_like_rate_%,100.0
1,like_rate_correct_%,100.0
2,like_rate_incorrect_%,100.0


=== Rate per Scale Value ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:116: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_scale = df.groupby("scale_value").apply(lambda g: pd.Series({


,scale_value,total_images,overall_like_rate_%,like_rate_correct_%,like_rate_incorrect_%
0,10,50.0,100.0,100.0,100.0
1,100,50.0,100.0,100.0,100.0
2,1000,50.0,100.0,100.0,100.0
3,10000,50.0,100.0,100.0,100.0
4,100000,50.0,100.0,100.0,100.0
5,1000000,50.0,100.0,100.0,100.0


=== Per Image Results ===


,image,num,variant,scale_value,prompt,answer
0,001_correct_10,001,correct,10,You are shown a social media post.\nYou can ei...,like
2,002_correct_10,002,correct,10,You are shown a social media post.\nYou can ei...,like
4,003_correct_10,003,correct,10,You are shown a social media post.\nYou can ei...,like
6,004_correct_10,004,correct,10,You are shown a social media post.\nYou can ei...,like
8,005_correct_10,005,correct,10,You are shown a social media post.\nYou can ei...,like
...,...,...,...,...,...,...
291,021_incorrect_1000000,021,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
293,022_incorrect_1000000,022,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
295,023_incorrect_1000000,023,incorrect,1000000,You are shown a social media post.\nYou can ei...,like
297,024_incorrect_1000000,024,incorrect,1000000,You are shown a social media post.\nYou can ei...,like


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/ministral-3-14b/outputs/e1_analysis_metrics.csv


## Approach 2 -- paired A/B forced choice, full 7x7 `metrics/realistic` disparity grid

In [13]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_mistral)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

✅ 001_correct0_vs_incorrect0 → liked correct (answered A)
✅ 002_correct0_vs_incorrect0 → liked correct (answered A)
✅ 003_correct0_vs_incorrect0 → liked correct (answered A)
✅ 004_correct0_vs_incorrect0 → liked correct (answered B)
✅ 005_correct0_vs_incorrect0 → liked correct (answered A)
✅ 006_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 007_correct0_vs_incorrect0 → liked correct (answered A)
✅ 008_correct0_vs_incorrect0 → liked correct (answered A)
✅ 009_correct0_vs_incorrect0 → liked correct (answered A)
✅ 010_correct0_vs_incorrect0 → liked correct (answered B)
✅ 011_correct0_vs_incorrect0 → liked correct (answered B)
✅ 012_correct0_vs_incorrect0 → liked correct (answered B)
✅ 013_correct0_vs_incorrect0 → liked correct (answered A)
✅ 014_correct0_vs_incorrect0 → liked correct (answered B)
✅ 015_correct0_vs_incorrect0 → liked correct (answered A)
✅ 016_correct0_vs_incorrect0 → liked incorrect (answered A)
✅ 017_correct0_vs_incorrect0 → liked correct (answered A)
✅ 018_corr

In [14]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")


Metrics paired A/B analysis: e1_results_metrics_paired
=== Overall Summary ===


,metric,value
0,overall_liked_correct_%,67.02
1,overall_liked_incorrect_%,32.98
2,invalid_answer_%,0.00


=== Liked Correct Rate per Scale Pair ===


/home/jovyan/conformity-llms-facebook-posts/experiments/e1/e1_utils/e1_analysis_optimized.py:163: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_pair = df.groupby("pair").apply(lambda g: pd.Series({


,pair,total_pairs,liked_correct_%,liked_incorrect_%,invalid_%
0,0_vs_0,25.0,92.0,8.0,0.0
1,0_vs_10,25.0,52.0,48.0,0.0
2,0_vs_100,25.0,44.0,56.0,0.0
3,0_vs_1000,25.0,32.0,68.0,0.0
4,0_vs_10000,25.0,28.0,72.0,0.0
5,0_vs_100000,25.0,56.0,44.0,0.0
6,0_vs_1000000,25.0,24.0,76.0,0.0
7,10_vs_0,25.0,100.0,0.0,0.0
8,10_vs_10,25.0,96.0,4.0,0.0
9,10_vs_100,25.0,24.0,76.0,0.0


=== Per Pair Results ===


,image,num,correct_scale,incorrect_scale,post_a_variant,post_b_variant,prompt,answer,liked_variant,pair
0,001_correct0_vs_incorrect0,001,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
1,002_correct0_vs_incorrect0,002,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
2,003_correct0_vs_incorrect0,003,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
3,004_correct0_vs_incorrect0,004,0,0,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,0_vs_0
4,005_correct0_vs_incorrect0,005,0,0,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,0_vs_0
...,...,...,...,...,...,...,...,...,...,...
1220,021_correct1000000_vs_incorrect1000000,021,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,1000000_vs_1000000
1221,022_correct1000000_vs_incorrect1000000,022,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",B,correct,1000000_vs_1000000
1222,023_correct1000000_vs_incorrect1000000,023,1000000,1000000,incorrect,correct,"You are shown two social media posts, Post A a...",A,incorrect,1000000_vs_1000000
1223,024_correct1000000_vs_incorrect1000000,024,1000000,1000000,correct,incorrect,"You are shown two social media posts, Post A a...",A,correct,1000000_vs_1000000


✅ Saved to: /home/jovyan/conformity-llms-facebook-posts/experiments/e1_climate/ministral-3-14b/outputs/e1_analysis_metrics_paired.csv
